Question 1: Less Features is More

In [ ]:
# # Recreate df from df_full to make sure all original columns are present
# df = df_full[df_full.Date >= '2000-01-01'].copy()

# # Convert Month and Weekday formatting as done in the original pipeline
# df.loc[:, 'Month'] = pd.to_datetime(df['Month'], format='%m').dt.strftime('%B')
# df.loc[:, 'Weekday'] = df['Weekday'].astype('string')

# # 1. Redefine CATEGORICAL by removing 'Ticker' and 'month_wom'
# new_categorical = ['Month', 'Weekday', 'ticker_type']

# # Generate dummy variables
# dummy_variables_new = pd.get_dummies(df[new_categorical], dtype='int32')
# DUMMIES_new = dummy_variables_new.keys().to_list()

# # Combine with original df
# df_with_dummies_new = pd.concat([df, dummy_variables_new], axis=1)

# # 2. Re-apply temporal split
# df_with_dummies_new = temporal_split(df_with_dummies_new,
#                                      min_date=min_date_df,
#                                      max_date=max_date_df)

# new_df_new = df_with_dummies_new.copy()

# # 3. Features to be used (numerical + new dummies)
# features_list_new = NUMERICAL + DUMMIES_new
# print(f"New features count: {len(features_list_new)}") # Expected: 208

# # 4. Prepare data sets
# to_predict = 'is_positive_growth_30d_future'
# train_valid_df_new = new_df_new[new_df_new.split.isin(['train','validation'])].copy(deep=True)
# test_df_new = new_df_new[new_df_new.split.isin(['test'])].copy(deep=True)

# X_train_valid_new = train_valid_df_new[features_list_new + [to_predict]]
# X_test_new = test_df_new[features_list_new + [to_predict]]
# X_all_new = new_df_new[features_list_new + [to_predict]].copy(deep=True)

# # Clean DataFrames
# X_train_valid_new = clean_dataframe_from_inf_and_nan(X_train_valid_new)
# X_test_new = clean_dataframe_from_inf_and_nan(X_test_new)
# X_all_new = clean_dataframe_from_inf_and_nan(X_all_new)

# y_train_valid_new = X_train_valid_new[to_predict]
# y_test_new = X_test_new[to_predict]
# y_all_new = X_all_new[to_predict]

# del X_train_valid_new[to_predict]
# del X_test_new[to_predict]
# del X_all_new[to_predict]

# # 5. Fit the Random Forest model
# from sklearn.ensemble import RandomForestClassifier
# rf_best_new = RandomForestClassifier(n_estimators=100,
#                                      max_depth=13,
#                                      random_state=42,
#                                      n_jobs=-1)
# rf_best_new.fit(X_train_valid_new, y_train_valid_new)

# # 6. Predict probabilities and apply threshold for pred10 (threshold = 0.55)
# y_pred_all_rf = rf_best_new.predict_proba(X_all_new)
# y_pred_all_class1_rf = np.array([k[1] for k in y_pred_all_rf])
# new_df_new['pred10_rf_best_rule_55'] = (y_pred_all_class1_rf >= 0.55).astype(int)

# # 7. Trading simulation for pred10_rf_best_rule_55
# pred_col = 'pred10_rf_best_rule_55'
# filter_test = (new_df_new.split == 'test') & (new_df_new[pred_col] == 1)

# sim1_count_investments = len(new_df_new[filter_test])
# gross_rev = (new_df_new[filter_test][pred_col] * 100 * (new_df_new[filter_test]['growth_future_30d'] - 1)).sum()
# fees = (-new_df_new[filter_test][pred_col] * 100 * 0.002).sum()
# net_rev = gross_rev + fees

# df_daily = pd.DataFrame(new_df_new[filter_test].groupby('Date')[pred_col].count())
# q75_daily = df_daily[pred_col].quantile(0.75)
# capital = 100 * 30 * q75_daily

# cagr = ((capital + net_rev) / capital) ** (1/4)

# print(f"=== New Setup Results for pred10_rf_best_rule_55 ===")
# print(f"Count investments on test: {sim1_count_investments}")
# print(f"Net Revenue: ${net_rev:.2f}")
# print(f"Capital Required (Q75 daily investments * 30 * 100): ${capital:.2f}")
# print(f"CAGR: {cagr:.4f} or {100*(cagr-1):.2f}%")

In [ ]:
'''

New features count: 208
=== New Setup Results for pred10_rf_best_rule_55 ===
Count investments on test: 3514
Net Revenue: $11962.11
Capital Required (Q75 daily investments * 30 * 100): $18000.00
CAGR: 1.1359 or 13.59%

'''



Question 2: Most Efficient Random Forest Decision Rule

In [ ]:
# import numpy as np
# import pandas as pd

# # 1. Predict probabilities on X_all and X_test using rf_best
# y_pred_all_rf = rf_best.predict_proba(X_all)
# y_pred_all_rf_class1 = np.array([k[1] for k in y_pred_all_rf])

# # 2. Iterate and define new predictors for thresholds 0.56 to 0.64
# thresholds = [0.56, 0.57, 0.58, 0.59, 0.60, 0.61, 0.62, 0.63, 0.64]
# for th in thresholds:
#     rule_suffix = f"{int(th*100)}"
#     pred_name = f'pred_rf_rule_{rule_suffix}'
    
#     new_df[pred_name] = (y_pred_all_rf_class1 >= th).astype(int)
    
#     # Setup simulation variables
#     new_df[f'sim1_gross_rev_rf_{rule_suffix}'] = new_df[pred_name] * 100 * (new_df['growth_future_30d'] - 1)
#     new_df[f'sim1_fees_rf_{rule_suffix}'] = -new_df[pred_name] * 100 * 0.002
#     new_df[f'sim1_net_rev_rf_{rule_suffix}'] = new_df[f'sim1_gross_rev_rf_{rule_suffix}'] + new_df[f'sim1_fees_rf_{rule_suffix}']

# # 3. Analyze results on test set
# results_rf = []
# for th in thresholds:
#     rule_suffix = f"{int(th*100)}"
#     pred_name = f'pred_rf_rule_{rule_suffix}'
    
#     filter_test_positive = (new_df.split == 'test') & (new_df[pred_name] == 1)
#     count_investments = filter_test_positive.sum()
#     net_rev = new_df[filter_test_positive][f'sim1_net_rev_rf_{rule_suffix}'].sum()
    
#     # Daily average calculations
#     daily_counts = new_df[filter_test_positive].groupby('Date')[pred_name].count()
#     avg_trades_per_day = daily_counts.mean() if len(daily_counts) > 0 else 0
#     q75_trades_per_day = daily_counts.quantile(0.75) if len(daily_counts) > 0 else 0
    
#     # Capital and CAGR
#     capital = 100 * 30 * q75_trades_per_day if q75_trades_per_day > 0 else 1
#     cagr = ((capital + net_rev) / capital) ** (1/4) if (capital + net_rev) > 0 else 0
    
#     results_rf.append({
#         'threshold': th,
#         'count_investments': count_investments,
#         'avg_trades_per_day': avg_trades_per_day,
#         'cagr': cagr,
#         'net_revenue': net_rev
#     })

# df_rf_analysis = pd.DataFrame(results_rf)
# print(df_rf_analysis.to_string(index=False))


In [ ]:
'''

threshold  count_investments  avg_trades_per_day     cagr  net_revenue
      0.56               1777            6.106529 1.066384  7036.001776
      0.57               1777            6.106529 1.066384  7036.001776
      0.58               1305            5.852018 1.044526  5139.641346
      0.59                906            4.950820 1.036852  3270.883816
      0.60                641            4.006250 1.029079  2186.816844
      0.61                406            3.075758 1.029481  1478.903921
      0.62                231            2.510870 1.031748  1198.520912
      0.63                134            2.196721 1.017961   664.217906
      0.64                 74            1.761905 1.018090   446.085037


'''

In [ ]:
# # 1. Redefine target variable based on the new threshold (> 1.022)
# new_df['is_positive_growth_30d_future'] = (new_df['growth_future_30d'] > 1.022).astype(int)

# # 2. Re-prepare training/validation/test sets with updated target
# train_valid_df = new_df[new_df.split.isin(['train','validation'])].copy(deep=True)
# test_df = new_df[new_df.split.isin(['test'])].copy(deep=True)

# X_train_valid = train_valid_df[features_list + ['is_positive_growth_30d_future']]
# X_all = new_df[features_list + ['is_positive_growth_30d_future']].copy(deep=True)

# X_train_valid = clean_dataframe_from_inf_and_nan(X_train_valid)
# X_all = clean_dataframe_from_inf_and_nan(X_all)

# y_train_valid = X_train_valid['is_positive_growth_30d_future']
# y_all = X_all['is_positive_growth_30d_future']

# del X_train_valid['is_positive_growth_30d_future']
# del X_all['is_positive_growth_30d_future']

# # 3. Fit Decision Tree Classifier (max_depth=10)
# clf_10, _ = fit_decision_tree(X=X_train_valid, y=y_train_valid, max_depth=10)

# # 4. Predict on full dataset and save as pred5_clf_10
# y_pred_all = clf_10.predict(X_all)
# new_df['pred5_clf_10'] = y_pred_all

# # 5. Count trades (sum of positive predictions) on the TEST set
# test_trades_count = new_df[new_df.split == 'test']['pred5_clf_10'].sum()
# print(f"Number of trades for the best predictor pred5_clf_10 on TEST set: {test_trades_count}")


In [ ]:
#Number of trades for the best predictor pred5_clf_10 on TEST set: 7

Question 4: [EXPLORATORY] Describe Your Ideal Trading Strategy